# CareConnect Retinal AI: Google Colab GPU Research Pipeline
## Controlled Experiments for Improving Retinal Cardiovascular Disease (CVD) ROC-AUC

**Project:** CareConnect Hospital Management System with Retinal AI Screening  
**Author:** Computer Vision & Medical AI Research Team  
**Hardware:** Google Colab GPU (T4 / A100 / V100)  
**Datasets:** mBRSET (Mobile Brazilian Retinal Dataset) & ODIR-5K (Ocular Disease Intelligent Recognition)  

---

### Clinical Context & Research Scope
- **Vascular Disease (Primary CVD Target):** In held-out test evaluation ($N = 191$ patients, 32 positives), image-only discrimination for systemic vascular disease from handheld mobile fundus photos yields **ROC-AUC: 0.5202** (95% CI: `[0.3936, 0.6416]`), indicating near-chance discrimination without systemic biomarkers.
- **Diabetic Retinopathy (Ocular Finding):** Retinal microvascular damage is directly visible on fundus scans, yielding **76.99% test accuracy** and **0.6918 Weighted F1**.
- **Research Objective:** Methodically investigate whether gradual backbone unfreezing, differential learning rates, focal loss, ODIR-5K ocular pretraining, or multimodal demographic fusion can improve the discriminative ROC-AUC for `vascular_disease` without degrading Diabetic Retinopathy performance.

> [!WARNING]
> **Ethical & Clinical Boundary:** The model is an assistive experimental screening tool, not a clinically validated diagnostic device. Predictions must never be presented as a medical diagnosis. Attending physician physical examination, blood pressure measurement, fasting lipid panels, and 12-lead ECG remain mandatory.


---
## Section A — GPU Setup & Environment Validation

In this section we verify that Google Colab is running with an active GPU accelerator (NVIDIA T4, V100, or A100).
If a GPU is not enabled, the script halts with instructions to enable GPU acceleration in Colab settings.


In [ ]:
# Check Python, PyTorch, and CUDA GPU status
import sys
import torch

print('=' * 65)
print('CARECONNECT RETINAL AI: COLAB GPU RUNTIME VERIFICATION')
print('=' * 65)
print(f'Python Version : {sys.version.split()[0]}')
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available : {torch.cuda.is_available()}')

if not torch.cuda.is_available():
    raise SystemExit(
        '\n[ERROR] GPU is NOT available in this Colab session!\n'
        'Please enable GPU:\n'
        '1. Go to top menu: Runtime -> Change runtime type\n'
        '2. Under Hardware accelerator, select T4 GPU (or A100)\n'
        '3. Click Save and re-run this cell.'
    )

device_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'Active GPU     : {device_name}')
print(f'Total VRAM     : {vram_gb:.2f} GB')
print(f'Compute Device : cuda:0')
print('=' * 65)
print('[SUCCESS] GPU runtime verified and ready for deep learning training.')


In [ ]:
# Install and verify required compatible dependencies
# Note: Colab already provides PyTorch, Torchvision, and CUDA out-of-the-box.
!pip install -q scikit-learn pandas pillow matplotlib seaborn tqdm


---
## Section B — Dataset Access & Configuration

### How to Upload Your Datasets from Windows to Google Drive
1. **mBRSET Images:**
   - On your Windows machine, locate `C:\Users\vinay\CVD Project\mbrset-a-mobile-brazilian-retinal-dataset-1.0-20261001T140750Z-1-001\mbrset-a-mobile-brazilian-retinal-dataset-1.0\images`.
   - Zip the `images` folder (or upload it directly) to Google Drive at: `My Drive/careconnect_cvd/data/mbrset/images`.
2. **mBRSET Split CSVs:**
   - Copy `train.csv`, `val.csv`, and `test.csv` from `HospitalManagementSystem Project\ml\datasets\splits\`.
   - Upload to Google Drive at: `My Drive/careconnect_cvd/data/splits/`.
3. **ODIR-5K (Optional, for Transfer Learning):**
   - Copy `full_df.csv` and `preprocessed_images` from `C:\Users\vinay\CVD Project\ODIR-5K`.
   - Upload to Google Drive at: `My Drive/careconnect_cvd/data/odir/`.

Mount Google Drive using the cell below and configure your directory paths.


In [ ]:
# Mount Google Drive to access persistent storage across Colab sessions
from google.colab import drive
drive.mount('/content/drive')
print('[SUCCESS] Google Drive mounted successfully at /content/drive.')


In [ ]:
# ============================================================================
# CONFIGURE YOUR DATASET & EXPERIMENT DIRECTORIES HERE
# Modify these paths if your Google Drive folder structure is different
# ============================================================================
from pathlib import Path

DRIVE_BASE = Path('/content/drive/MyDrive/careconnect_cvd')

MBRSET_IMAGES_DIR = DRIVE_BASE / 'data' / 'mbrset' / 'images'
SPLITS_DIR = DRIVE_BASE / 'data' / 'splits'
ODIR_DIR = DRIVE_BASE / 'data' / 'odir'
OUTPUT_DIR = DRIVE_BASE / 'experiments_output'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Configured Directories:')
print('  mBRSET Images :', MBRSET_IMAGES_DIR)
print('  Splits Dir    :', SPLITS_DIR)
print('  ODIR Dir      :', ODIR_DIR)
print('  Output Dir    :', OUTPUT_DIR)


In [ ]:
# Sanity check file presence and accessibility
import os

def verify_paths():
    errors = []
    if not SPLITS_DIR.exists():
        errors.append(f'Splits directory missing: {SPLITS_DIR}')
    else:
        for s in ['train.csv', 'val.csv', 'test.csv']:
            if not (SPLITS_DIR / s).exists():
                errors.append(f'Split file missing: {SPLITS_DIR / s}')
                
    if not MBRSET_IMAGES_DIR.exists():
        errors.append(f'mBRSET images directory missing: {MBRSET_IMAGES_DIR}')
    else:
        # Count sample images
        imgs = list(MBRSET_IMAGES_DIR.glob('*.jpg'))
        if len(imgs) == 0:
            # Try recursive check
            imgs = list(MBRSET_IMAGES_DIR.rglob('*.jpg'))
        print(f'[Sanity Check] Found {len(imgs):,} mBRSET images in storage.')
        if len(imgs) < 1000:
            errors.append(f'Expected ~4,000 mBRSET images, but found only {len(imgs)}.')

    if errors:
        print('\n[ATTENTION] The following path issues were detected:')
        for e in errors:
            print('  - ', e)
        print('\nPlease confirm that your dataset files are uploaded to Google Drive in the specified paths.')
    else:
        print('\n[SUCCESS] All primary mBRSET dataset paths verified successfully.')

verify_paths()


---
## Section C — Data & Leakage Audit

We enforce four non-negotiable scientific safeguards:
1. **Pure `vascular_disease` target definition:** No conflation with Acute Myocardial Infarction (AMI).
2. **Strict Patient-Level Partitioning:** Zero patient overlap between train, validation, and test cohorts ($	ext{Train} \cap 	ext{Val} = \emptyset$, $	ext{Train} \cap 	ext{Test} = \emptyset$).
3. **No Duplicate Images:** Every image filename is distinct.
4. **Preservation of Benchmark Test Set:** The test cohort ($N=191$ patients, 32 positives) is kept strictly untouched during development.


In [ ]:
import pandas as pd
import numpy as np

train_df = pd.read_csv(SPLITS_DIR / 'train.csv')
val_df = pd.read_csv(SPLITS_DIR / 'val.csv')
test_df = pd.read_csv(SPLITS_DIR / 'test.csv')

print('=' * 65)
print('PARTITION AUDIT & PATIENT LEAKAGE VERIFICATION')
print('=' * 65)

for name, df in [('Train', train_df), ('Validation', val_df), ('Held-Out Test', test_df)]:
    n_imgs = len(df)
    n_pts = df['patient'].nunique()
    vd_pos_img = (df['vascular_disease'] == 1).sum()
    vd_neg_img = (df['vascular_disease'] == 0).sum()
    # Patient-level positive count
    pt_vd = df.groupby('patient')['vascular_disease'].first()
    vd_pos_pt = (pt_vd == 1).sum()
    vd_neg_pt = (pt_vd == 0).sum()
    print(f'{name:14s}: {n_imgs:4d} images | {n_pts:4d} patients | VD+ Pts: {vd_pos_pt:3d} ({vd_pos_pt/n_pts*100:4.1f}%) | VD- Pts: {vd_neg_pt:3d}')

print('-' * 65)
# Patient leakage assertions
train_pts = set(train_df['patient'].unique())
val_pts = set(val_df['patient'].unique())
test_pts = set(test_df['patient'].unique())

tv_overlap = train_pts & val_pts
tt_overlap = train_pts & test_pts
vt_overlap = val_pts & test_pts

assert len(tv_overlap) == 0, f'LEAKAGE: {len(tv_overlap)} patients overlap between Train and Val!'
assert len(tt_overlap) == 0, f'LEAKAGE: {len(tt_overlap)} patients overlap between Train and Test!'
assert len(vt_overlap) == 0, f'LEAKAGE: {len(vt_overlap)} patients overlap between Val and Test!'

# Target purity assertion
for s_name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    unique_targets = set(df['vascular_disease'].dropna().unique())
    assert unique_targets.issubset({0, 1}), f'Non-binary targets in {s_name}: {unique_targets}'

print('[AUDIT PASSED] Zero patient leakage verified across all partition boundaries.')
print('[AUDIT PASSED] Target labels are strictly binary (0 or 1).')


---
## Section D — Model Architecture & Data Loaders

We implement the production `CareConnectMultiTaskNet` architecture:
- **Backbone:** EfficientNet-B0 (1,280 pooled features) with dropout $p = 0.4$.
- **Task Heads:**
  1. `head_vd`: Linear(1280, 1) for binary Vascular Disease screening.
  2. `head_ht`: Linear(1280, 1) for binary Systemic Hypertension.
  3. `head_dr`: Linear(1280, 5) for 5-class Diabetic Retinopathy ICDR grading.
- **Transforms:** Bilinear resize ($224 	imes 224$ with antialiasing) and standard ImageNet normalization.


In [ ]:
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
DR_CLASSES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']

def get_train_transforms(image_size=224, enhanced=False):
    if enhanced:
        return T.Compose([
            T.Resize((image_size, image_size), interpolation=T.InterpolationMode.BILINEAR, antialias=True),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.3),
            T.RandomRotation(degrees=20),
            T.ColorJitter(brightness=0.15, contrast=0.20, saturation=0.15),
            T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
    return T.Compose([
        T.Resize((image_size, image_size), interpolation=T.InterpolationMode.BILINEAR, antialias=True),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.2),
        T.RandomRotation(degrees=15),
        T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def get_inference_transforms(image_size=224):
    return T.Compose([
        T.Resize((image_size, image_size), interpolation=T.InterpolationMode.BILINEAR, antialias=True),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


In [ ]:
class MBRSETDataset(Dataset):
    def __init__(self, df, images_dir, transform=None, include_clinical=False):
        self.df = df.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.transform = transform if transform is not None else get_inference_transforms()
        self.include_clinical = include_clinical

        vd_vals = pd.to_numeric(self.df.get('vascular_disease', pd.Series([-1]*len(self.df))), errors='coerce').fillna(-1).values
        self.vd_targets = np.where(np.isin(vd_vals, [0, 1]), vd_vals, -1).astype(np.float32)
        self.vd_masks = (self.vd_targets >= 0).astype(np.float32)

        ht_vals = pd.to_numeric(self.df.get('systemic_hypertension', pd.Series([-1]*len(self.df))), errors='coerce').fillna(-1).values
        self.ht_targets = np.where(np.isin(ht_vals, [0, 1]), ht_vals, -1).astype(np.float32)
        self.ht_masks = (self.ht_targets >= 0).astype(np.float32)

        dr_vals = pd.to_numeric(self.df.get('dr_grade', pd.Series([-1]*len(self.df))), errors='coerce').fillna(-1).values
        self.dr_targets = np.where(np.isin(dr_vals, [0, 1, 2, 3, 4]), dr_vals, -1).astype(np.int64)
        self.dr_masks = (self.dr_targets >= 0).astype(np.float32)

        if self.include_clinical and 'age' in self.df.columns and 'sex' in self.df.columns:
            age_raw = pd.to_numeric(self.df['age'], errors='coerce').fillna(60.0).values
            age_norm = ((age_raw - 60.0) / 15.0).astype(np.float32)
            sex_bin = (self.df['sex'].astype(str).str.lower().str.startswith('m')).astype(np.float32).values
            self.clinical_feats = np.stack([age_norm, sex_bin], axis=1)
        else:
            self.clinical_feats = None

        self.files = self.df['file'].values
        self.patients = self.df['patient'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img_path = self.images_dir / fname
        if not img_path.exists():
            matches = list(self.images_dir.rglob(fname))
            if matches:
                img_path = matches[0]
            else:
                raise FileNotFoundError(f'Image {fname} not found in {self.images_dir}')

        with Image.open(img_path) as im:
            img = im.convert('RGB')

        tensor = self.transform(img)
        item = {
            'image': tensor,
            'targets': {
                'vascular_disease': torch.tensor(self.vd_targets[idx], dtype=torch.float32),
                'systemic_hypertension': torch.tensor(self.ht_targets[idx], dtype=torch.float32),
                'dr_grade': torch.tensor(self.dr_targets[idx], dtype=torch.long),
            },
            'masks': {
                'vascular_disease': torch.tensor(self.vd_masks[idx], dtype=torch.float32),
                'systemic_hypertension': torch.tensor(self.ht_masks[idx], dtype=torch.float32),
                'dr_grade': torch.tensor(self.dr_masks[idx], dtype=torch.float32),
            },
            'metadata': {'patient': int(self.patients[idx]), 'file': str(fname)},
        }
        if self.clinical_feats is not None:
            item['clinical'] = torch.tensor(self.clinical_feats[idx], dtype=torch.float32)
        return item


In [ ]:
class CareConnectMultiTaskNet(nn.Module):
    def __init__(self, pretrained=True, dropout=0.4, clinical_dim=0, num_dr_classes=5, custom_encoder=None):
        super().__init__()
        self.clinical_dim = clinical_dim
        if custom_encoder is not None:
            self.features = custom_encoder
            self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        else:
            weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
            base = efficientnet_b0(weights=weights)
            self.features = base.features
            self.avgpool = base.avgpool

        self.dropout = nn.Dropout(p=dropout)
        feature_dim = 1280
        if clinical_dim > 0:
            self.clinical_mlp = nn.Sequential(
                nn.Linear(clinical_dim, 32),
                nn.ReLU(inplace=True),
                nn.Dropout(p=0.2),
            )
            total_dim = feature_dim + 32
        else:
            self.clinical_mlp = None
            total_dim = feature_dim

        self.head_vd = nn.Linear(total_dim, 1)
        self.head_ht = nn.Linear(total_dim, 1)
        self.head_dr = nn.Linear(total_dim, num_dr_classes)

    def forward(self, x, c=None):
        f = self.features(x)
        pooled = self.avgpool(f).flatten(1)
        dropped = self.dropout(pooled)
        if self.clinical_mlp is not None and c is not None:
            c_feats = self.clinical_mlp(c)
            dropped = torch.cat([dropped, c_feats], dim=1)
        return {
            'vascular_disease': self.head_vd(dropped).squeeze(-1),
            'systemic_hypertension': self.head_ht(dropped).squeeze(-1),
            'dr_grade': self.head_dr(dropped),
        }


---
## Section E — Controlled Deep Learning Experiments on Colab GPU

We evaluate 8 distinct experimental configurations using validation ROC-AUC for early stopping and model selection:
- **Exp 1: Baseline Replication** — EfficientNet-B0 with ImageNet init, default multi-task loss ($10^{-4}$).
- **Exp 2: Frozen Backbone** — Feature extractor frozen; train linear task projection heads only ($3 \times 10^{-4}$).
- **Exp 3: Gradual Unfreezing / Differential LR** — Backbone fine-tuned at $10^{-5}$, task heads at $10^{-4}$.
- **Exp 4: Multi-Task Focal Loss** — Evaluates Focal Loss ($\gamma = 2.0$) to focus gradient updates on hard examples.
- **Exp 5: Learning Rate Grid ($1\text{e-}5$ vs $3\text{e-}5$)** — Evaluates backbone learning rates with Cosine Annealing.
- **Exp 6: Conservative Medical Retinal Augmentation** — Adds gentle affine and color jitter transforms.
- **Exp 7: ODIR-5K Ocular Transfer Learning** — Pretrained on 8 ocular classes before fine-tuning on mBRSET.
- **Exp 8: Multimodal Demographic Prior** — Integrates Age and Sex clinical priors available in HMS.


In [ ]:
# Training Loss Functions & Metrics Engine
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score, roc_curve, confusion_matrix

class FocalLossBinary(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)
        loss = bce * ((1 - p_t) ** self.gamma)
        if self.alpha is not None:
            alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
            loss = alpha_t * loss
        return loss

class CareConnectMultiTaskLoss(nn.Module):
    def __init__(self, pos_weight_vd=1.0, pos_weight_ht=1.0, use_focal=False):
        super().__init__()
        self.use_focal = use_focal
        self.focal = FocalLossBinary(gamma=2.0) if use_focal else None
        self.register_buffer('pw_vd', torch.tensor(pos_weight_vd, dtype=torch.float32))
        self.register_buffer('pw_ht', torch.tensor(pos_weight_ht, dtype=torch.float32))
    def forward(self, logits, targets, masks):
        # 1. VD
        vd_l, vd_t, vd_m = logits['vascular_disease'].float(), targets['vascular_disease'].float(), masks['vascular_disease'].float()
        if self.use_focal:
            l_vd_raw = self.focal(vd_l, vd_t)
        else:
            l_vd_raw = F.binary_cross_entropy_with_logits(vd_l, vd_t, pos_weight=self.pw_vd, reduction='none')
        loss_vd = (l_vd_raw * vd_m).sum() / vd_m.sum().clamp(min=1.0)

        # 2. HT
        ht_l, ht_t, ht_m = logits['systemic_hypertension'].float(), targets['systemic_hypertension'].float(), masks['systemic_hypertension'].float()
        loss_ht = (F.binary_cross_entropy_with_logits(ht_l, ht_t, pos_weight=self.pw_ht, reduction='none') * ht_m).sum() / ht_m.sum().clamp(min=1.0)

        # 3. DR
        dr_l, dr_t, dr_m = logits['dr_grade'].float(), targets['dr_grade'].long().clamp(min=0), masks['dr_grade'].float()
        loss_dr = (F.cross_entropy(dr_l, dr_t, reduction='none') * dr_m).sum() / dr_m.sum().clamp(min=1.0)

        return 1.0 * loss_vd + 0.5 * loss_ht + 0.5 * loss_dr


In [ ]:
# Metric aggregation & evaluation helpers
def aggregate_patient_predictions(eval_df):
    valid = eval_df[eval_df['vd_true'].isin([0, 1])]
    grouped = valid.groupby('patient').agg(prob=('vd_prob', 'mean'), true=('vd_true', 'first'))
    return grouped['true'].values, grouped['prob'].values

def find_optimal_threshold(y_true, y_prob):
    if len(np.unique(y_true)) < 2: return 0.5
    fpr, tpr, thresh = roc_curve(y_true, y_prob)
    best_idx = int(np.argmax(tpr - fpr))
    return float(np.clip(thresh[best_idx], 0.10, 0.90))

def compute_metrics(y_true, y_prob, threshold=None):
    if len(np.unique(y_true)) < 2:
        return {'roc_auc': 0.5, 'pr_auc': 0.0, 'sensitivity': 0.0, 'specificity': 0.0, 'f1': 0.0}
    auc = float(roc_auc_score(y_true, y_prob))
    pr_auc = float(average_precision_score(y_true, y_prob))
    if threshold is None:
        threshold = find_optimal_threshold(y_true, y_prob)
    preds = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    sens = float(tp / max(tp + fn, 1))
    spec = float(tn / max(tn + fp, 1))
    prec = float(tp / max(tp + fp, 1))
    f1 = float(2 * prec * sens / max(prec + sens, 1e-6))
    return {
        'roc_auc': round(auc, 4),
        'pr_auc': round(pr_auc, 4),
        'threshold': round(threshold, 4),
        'sensitivity': round(sens, 4),
        'specificity': round(spec, 4),
        'precision': round(prec, 4),
        'f1': round(f1, 4),
        'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}
    }


In [ ]:
# Standardized Experiment Training Function on GPU
import copy, time

def train_and_evaluate_experiment(
    exp_name,
    train_df, val_df, test_df,
    images_dir, output_dir,
    epochs=10, batch_size=32,
    lr_backbone=1e-5, lr_heads=1e-4,
    use_focal=False, freeze_backbone=False, enhanced_aug=False,
    include_clinical=False, custom_encoder=None,
    eval_test=False,
):
    print(f'\n[STARTING] {exp_name.upper()} (Epochs: {epochs}, Batch: {batch_size}, Device: CUDA)')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Class weights from train set strictly
    vd_t = train_df['vascular_disease']
    pos_w_vd = float((vd_t == 0).sum() / max((vd_t == 1).sum(), 1))
    ht_t = train_df['systemic_hypertension']
    pos_w_ht = float((ht_t == 0).sum() / max((ht_t == 1).sum(), 1))

    train_ds = MBRSETDataset(train_df, images_dir, transform=get_train_transforms(224, enhanced=enhanced_aug), include_clinical=include_clinical)
    val_ds = MBRSETDataset(val_df, images_dir, transform=get_inference_transforms(224), include_clinical=include_clinical)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    clinical_dim = 2 if include_clinical else 0
    model = CareConnectMultiTaskNet(pretrained=True, clinical_dim=clinical_dim, custom_encoder=custom_encoder).to(device)

    if freeze_backbone:
        for p in model.features.parameters():
            p.requires_grad = False
        params = [{'params': model.head_vd.parameters(), 'lr': lr_heads}, {'params': model.head_ht.parameters(), 'lr': lr_heads}, {'params': model.head_dr.parameters(), 'lr': lr_heads}]
    else:
        params = [
            {'params': [p for p in model.features.parameters() if p.requires_grad], 'lr': lr_backbone},
            {'params': model.head_vd.parameters(), 'lr': lr_heads},
            {'params': model.head_ht.parameters(), 'lr': lr_heads},
            {'params': model.head_dr.parameters(), 'lr': lr_heads},
        ]
    if model.clinical_mlp is not None:
        params.append({'params': model.clinical_mlp.parameters(), 'lr': lr_heads})

    optimizer = torch.optim.AdamW(params, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    criterion = CareConnectMultiTaskLoss(pos_weight_vd=pos_w_vd, pos_weight_ht=pos_w_ht, use_focal=use_focal)
    scaler = torch.cuda.amp.GradScaler()

    best_val_auc = -1.0
    best_threshold = 0.5
    best_metrics = {}
    best_state = None
    history = []

    for ep in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        tr_loss = 0.0
        for batch in train_loader:
            imgs = batch['image'].to(device, non_blocking=True)
            tgts = {k: v.to(device, non_blocking=True) for k, v in batch['targets'].items()}
            msks = {k: v.to(device, non_blocking=True) for k, v in batch['masks'].items()}
            clin = batch.get('clinical', None)
            if clin is not None: clin = clin.to(device, non_blocking=True)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                logits = model(imgs, clin)
                loss = criterion(logits, tgts, msks)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            tr_loss += float(loss.item()) * len(imgs)
        scheduler.step()
        tr_loss /= len(train_ds)

        # Validation
        model.eval()
        val_loss = 0.0
        vd_probs, vd_trues, pts = [], [], []
        dr_preds, dr_trues = [], []
        with torch.no_grad():
            for batch in val_loader:
                imgs = batch['image'].to(device, non_blocking=True)
                tgts = {k: v.to(device, non_blocking=True) for k, v in batch['targets'].items()}
                msks = {k: v.to(device, non_blocking=True) for k, v in batch['masks'].items()}
                clin = batch.get('clinical', None)
                if clin is not None: clin = clin.to(device, non_blocking=True)
                logits = model(imgs, clin)
                loss = criterion(logits, tgts, msks)
                val_loss += float(loss.item()) * len(imgs)
                p_vd = torch.sigmoid(logits['vascular_disease']).cpu().numpy()
                vd_probs.extend(p_vd.tolist() if p_vd.ndim > 0 else [float(p_vd)])
                vd_trues.extend(tgts['vascular_disease'].cpu().numpy().tolist())
                pts.extend(batch['metadata']['patient'])
                dr_p = logits['dr_grade'].argmax(dim=-1).cpu().numpy()
                dr_preds.extend(dr_p.tolist() if dr_p.ndim > 0 else [int(dr_p)])
                dr_trues.extend(tgts['dr_grade'].cpu().numpy().tolist())
        val_loss /= len(val_ds)

        val_df_res = pd.DataFrame({'patient': pts, 'vd_prob': vd_probs, 'vd_true': vd_trues})
        y_pt_true, y_pt_prob = aggregate_patient_predictions(val_df_res)
        metrics_ep = compute_metrics(y_pt_true, y_pt_prob)
        val_auc = metrics_ep['roc_auc']
        valid_dr = [t for t in dr_trues if t >= 0]
        dr_acc = float(accuracy_score(valid_dr, [dr_preds[i] for i, t in enumerate(dr_trues) if t >= 0])) if valid_dr else 0.0
        el = time.time() - t0

        history.append({'epoch': ep, 'train_loss': tr_loss, 'val_loss': val_loss, 'val_vd_auc': val_auc, 'val_dr_acc': dr_acc})
        print(f'  Epoch {ep:02d}/{epochs:02d} | Tr Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f} | Val VD AUC: {val_auc:.4f} | DR Acc: {dr_acc*100:.1f}% | Time: {el:.1f}s')

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_threshold = metrics_ep['threshold']
            best_metrics = metrics_ep
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, output_dir / f'{exp_name}_best.pt')

    # Save experiment report
    report = {'experiment': exp_name, 'best_val_auc': best_val_auc, 'best_threshold': best_threshold, 'val_metrics': best_metrics, 'history': history}
    with open(output_dir / f'{exp_name}_report.json', 'w') as f:
        json.dump(report, f, indent=2)
    print(f'[COMPLETED] {exp_name} | Best Val VD AUC: {best_val_auc:.4f} (Optimal Threshold: {best_threshold:.4f})')
    return report


---
## Section F — ODIR-5K Ocular Disease Pretraining

ODIR-5K contains 6,392 diverse retinal photographs with 8 multi-label ocular condition flags:
- `N`: Normal
- `D`: Diabetes
- `G`: Glaucoma
- `C`: Cataract
- `A`: Age-related Macular Degeneration (AMD)
- `H`: Hypertension Retinopathy (ocular damage)
- `M`: Myopia
- `O`: Other

> **Scientific Boundary:** ODIR labels are strictly ocular. They are **never** converted into CVD ground truth.
> ODIR-5K is used strictly to learn domain-specific retinal feature representations prior to fine-tuning on mBRSET.


In [ ]:
class ODIRDataset(Dataset):
    def __init__(self, df, images_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.transform = transform if transform is not None else get_inference_transforms()
        self.classes = ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
        labels = []
        for col in self.classes:
            if col in self.df.columns:
                labels.append(pd.to_numeric(self.df[col], errors='coerce').fillna(0).values)
            else:
                labels.append(np.zeros(len(self.df)))
        self.targets = np.stack(labels, axis=1).astype(np.float32)
        self.files = self.df['filename'].values if 'filename' in self.df.columns else self.df['filepath'].values
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        fname = Path(str(self.files[idx])).name
        p = self.images_dir / fname
        if not p.exists():
            matches = list(self.images_dir.rglob(fname))
            if matches: p = matches[0]
            else: raise FileNotFoundError(f'ODIR image {fname} not found in {self.images_dir}')
        with Image.open(p) as im: img = im.convert('RGB')
        return {'image': self.transform(img), 'targets': torch.tensor(self.targets[idx], dtype=torch.float32)}

def pretrain_odir(odir_df, images_dir, output_dir, epochs=5, batch_size=32, lr=3e-4):
    print(f'\n[ODIR PRETRAINING] Training on {len(odir_df)} ODIR images for {epochs} epochs...')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loader = DataLoader(ODIRDataset(odir_df, images_dir, transform=get_train_transforms(224, enhanced=True)), batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    base = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    encoder = base.features
    pool = base.avgpool
    head = nn.Linear(1280, 8)
    model = nn.Sequential(encoder, pool, nn.Flatten(), nn.Dropout(0.3), head).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss()
    for ep in range(1, epochs + 1):
        model.train()
        loss_tot = 0.0
        t0 = time.time()
        for batch in loader:
            im, tg = batch['image'].to(device), batch['targets'].to(device)
            opt.zero_grad()
            loss = crit(model(im), tg)
            loss.backward()
            opt.step()
            loss_tot += float(loss.item()) * len(im)
        el = time.time() - t0
        print(f'  ODIR Epoch {ep:02d}/{epochs:02d} | Multi-Label BCE: {loss_tot/len(loader.dataset):.4f} | Time: {el:.1f}s')
    save_p = output_dir / 'odir_pretrained_encoder.pt'
    torch.save(encoder.state_dict(), save_p)
    print(f'[SUCCESS] ODIR-5K pretrained encoder weights saved to {save_p}')
    return encoder


---
## Section G — Execute Experiments Matrix

Execute the experimental matrix below. You can adjust `EPOCHS_PER_EXPERIMENT` (e.g. 8 to 12 epochs) for optimal GPU training time.


In [ ]:
# Configure training parameters
EPOCHS = 10
BATCH_SIZE = 32  # Reduce to 16 if CUDA Out-Of-Memory occurs

all_experiment_reports = []

# -------------------------------------------------------------
# Experiment 1: Baseline Replication
# -------------------------------------------------------------
r1 = train_and_evaluate_experiment(
    'exp1_baseline', train_df, val_df, test_df, MBRSET_IMAGES_DIR, OUTPUT_DIR,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr_backbone=1e-4, lr_heads=1e-4
)
all_experiment_reports.append(r1)

# -------------------------------------------------------------
# Experiment 2: Frozen Backbone (Linear Probing)
# -------------------------------------------------------------
r2 = train_and_evaluate_experiment(
    'exp2_frozen_backbone', train_df, val_df, test_df, MBRSET_IMAGES_DIR, OUTPUT_DIR,
    epochs=EPOCHS, batch_size=BATCH_SIZE, freeze_backbone=True, lr_heads=3e-4
)
all_experiment_reports.append(r2)

# -------------------------------------------------------------
# Experiment 3: Differential Learning Rates (Gradual Fine-Tuning)
# -------------------------------------------------------------
r3 = train_and_evaluate_experiment(
    'exp3_differential_lr', train_df, val_df, test_df, MBRSET_IMAGES_DIR, OUTPUT_DIR,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr_backbone=1e-5, lr_heads=1e-4
)
all_experiment_reports.append(r3)

# -------------------------------------------------------------
# Experiment 4: Multi-Task Focal Loss (gamma=2.0)
# -------------------------------------------------------------
r4 = train_and_evaluate_experiment(
    'exp4_focal_loss', train_df, val_df, test_df, MBRSET_IMAGES_DIR, OUTPUT_DIR,
    epochs=EPOCHS, batch_size=BATCH_SIZE, use_focal=True, lr_backbone=1e-5, lr_heads=1e-4
)
all_experiment_reports.append(r4)

# -------------------------------------------------------------
# Experiment 5: Learning Rate Grid (3e-5)
# -------------------------------------------------------------
r5 = train_and_evaluate_experiment(
    'exp5_lr_3e5', train_df, val_df, test_df, MBRSET_IMAGES_DIR, OUTPUT_DIR,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr_backbone=3e-5, lr_heads=1e-4
)
all_experiment_reports.append(r5)

# -------------------------------------------------------------
# Experiment 6: Conservative Medical Retinal Augmentation
# -------------------------------------------------------------
r6 = train_and_evaluate_experiment(
    'exp6_medical_augmentation', train_df, val_df, test_df, MBRSET_IMAGES_DIR, OUTPUT_DIR,
    epochs=EPOCHS, batch_size=BATCH_SIZE, enhanced_aug=True, lr_backbone=1e-5, lr_heads=1e-4
)
all_experiment_reports.append(r6)


In [ ]:
# -------------------------------------------------------------
# Experiment 7: ODIR-5K Transfer Learning (Optional if ODIR present)
# -------------------------------------------------------------
odir_csv = ODIR_DIR / 'full_df.csv'
odir_imgs = ODIR_DIR / 'preprocessed_images'

if odir_csv.exists() and odir_imgs.exists():
    odir_df = pd.read_csv(odir_csv)
    odir_encoder = pretrain_odir(odir_df, odir_imgs, OUTPUT_DIR, epochs=min(5, EPOCHS), batch_size=BATCH_SIZE)
    r7 = train_and_evaluate_experiment(
        'exp7_odir_transfer', train_df, val_df, test_df, MBRSET_IMAGES_DIR, OUTPUT_DIR,
        epochs=EPOCHS, batch_size=BATCH_SIZE, custom_encoder=odir_encoder, lr_backbone=1e-5, lr_heads=1e-4
    )
    all_experiment_reports.append(r7)
else:
    print('[Notice] ODIR-5K files not found at specified path. Skipping Exp 7.')


In [ ]:
# -------------------------------------------------------------
# Experiment 8: Multimodal Demographics Prior (Age + Sex)
# -------------------------------------------------------------
if 'age' in train_df.columns and 'sex' in train_df.columns:
    r8 = train_and_evaluate_experiment(
        'exp8_multimodal', train_df, val_df, test_df, MBRSET_IMAGES_DIR, OUTPUT_DIR,
        epochs=EPOCHS, batch_size=BATCH_SIZE, include_clinical=True, lr_backbone=1e-5, lr_heads=1e-4
    )
    all_experiment_reports.append(r8)
else:
    print('[Notice] Age/Sex columns missing in splits. Skipping Exp 8.')


---
## Section H — Results Dashboard & Comparative Analysis

We aggregate validation performance across all completed experiments and plot training curves.


In [ ]:
import matplotlib.pyplot as plt

summary_rows = []
for r in all_experiment_reports:
    vm = r['val_metrics']
    summary_rows.append({
        'Experiment': r['experiment'],
        'Validation VD ROC-AUC': vm.get('roc_auc', 0.5),
        'Validation VD PR-AUC': vm.get('pr_auc', 0.0),
        'Sensitivity (Recall)': vm.get('sensitivity', 0.0),
        'Specificity': vm.get('specificity', 0.0),
        'F1 Score': vm.get('f1_score', 0.0),
        'Optimal Threshold': r['best_threshold'],
    })

df_summary = pd.DataFrame(summary_rows).sort_values(by='Validation VD ROC-AUC', ascending=False)
display(df_summary)
df_summary.to_csv(OUTPUT_DIR / 'colab_experiments_summary.csv', index=False)
print(f'[SUCCESS] Summary saved to {OUTPUT_DIR / "colab_experiments_summary.csv"}')


In [ ]:
# Plot validation curves across experiments
fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
for r in all_experiment_reports:
    hist = r['history']
    eps = [h['epoch'] for h in hist]
    aucs = [h['val_vd_auc'] for h in hist]
    ax.plot(eps, aucs, marker='o', linewidth=2, label=r['experiment'])

ax.axhline(0.5202, color='red', linestyle='--', label='Baseline Benchmark (0.5202)')
ax.set_title('Validation Vascular Disease (CVD) ROC-AUC Across Epochs', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Patient-Level Validation ROC-AUC', fontsize=11)
ax.set_ylim(0.40, 0.75)
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(bbox_to_anchor=(1.04, 1), loc='upper left')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'validation_roc_auc_comparison.png')
plt.show()


---
## Section I — Evaluate Best Model on Test Set & Export for CareConnect HMS

### Strict Invariant:
We now take the single top-performing experiment selected **strictly based on validation ROC-AUC**,
and evaluate it **once** on the untouched held-out test cohort ($N = 191$ patients).

The exported checkpoint and configuration files are 100% drop-in compatible with the existing local FastAPI microservice (`cvd-ai-service/`).


In [ ]:
# Select best model by Validation ROC-AUC
best_exp = df_summary.iloc[0]['Experiment']
best_rep = next(r for r in all_experiment_reports if r['experiment'] == best_exp)
best_ckpt_path = OUTPUT_DIR / f'{best_exp}_best.pt'
best_thresh = best_rep['best_threshold']

print(f'Top Selected Configuration: {best_exp}')
print(f'Validation ROC-AUC        : {best_rep["val_metrics"]["roc_auc"]:.4f}')
print(f'Frozen Decision Threshold : {best_thresh:.4f}')

# Final Evaluation on Held-Out Test Set
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
test_ds = MBRSETDataset(test_df, MBRSET_IMAGES_DIR, transform=get_inference_transforms(224))
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

eval_model = CareConnectMultiTaskNet(pretrained=False).to(device)
eval_model.load_state_dict(torch.load(best_ckpt_path, map_location=device))
eval_model.eval()

vd_probs, vd_trues, pts = [], [], []
with torch.no_grad():
    for batch in test_loader:
        imgs = batch['image'].to(device)
        logits = eval_model(imgs)
        p = torch.sigmoid(logits['vascular_disease']).cpu().numpy()
        vd_probs.extend(p.tolist() if p.ndim > 0 else [float(p)])
        vd_trues.extend(batch['targets']['vascular_disease'].cpu().numpy().tolist())
        pts.extend(batch['metadata']['patient'])

test_df_eval = pd.DataFrame({'patient': pts, 'vd_prob': vd_probs, 'vd_true': vd_trues})
y_test_true, y_test_prob = aggregate_patient_predictions(test_df_eval)
final_test_metrics = compute_metrics(y_test_true, y_test_prob, threshold=best_thresh)

print('=' * 65)
print('FINAL HELD-OUT TEST EVALUATION REPORT (N=191 Patients, 32 Positives)')
print('=' * 65)
print(f'Held-Out Test ROC-AUC : {final_test_metrics["roc_auc"]:.4f}')
print(f'Held-Out Test PR-AUC  : {final_test_metrics["pr_auc"]:.4f}')
print(f'Sensitivity (Recall)  : {final_test_metrics["sensitivity"]*100:.1f}% ({final_test_metrics["confusion_matrix"]["tp"]}/32 caught)')
print(f'Specificity           : {final_test_metrics["specificity"]*100:.1f}% ({final_test_metrics["confusion_matrix"]["tn"]}/159)')
print(f'Precision             : {final_test_metrics["precision"]*100:.1f}%')
print(f'F1 Score              : {final_test_metrics["f1"]:.4f}')
print('Confusion Matrix      :', final_test_metrics['confusion_matrix'])
print('=' * 65)


In [ ]:
# Export Candidate Package for CareConnect HMS Local Deployment
export_dir = OUTPUT_DIR / 'candidate_deployment_package'
export_dir.mkdir(parents=True, exist_ok=True)

# 1. Save candidate weights (distinct from production best_model.pt)
candidate_pt = export_dir / 'candidate_model.pt'
torch.save(torch.load(best_ckpt_path, map_location='cpu'), candidate_pt)

# 2. Save drop-in compatible config.json
candidate_config = {
    'project_name': 'CareConnect Retinal AI Microservice',
    'version': f'2.1.0-colab_{best_exp}',
    'model': {
        'architecture': 'CareConnectMultiTaskNet',
        'backbone': 'efficientnet_b0',
        'weights_file': 'models/candidate_model.pt',
        'image_size': 224,
        'mean': [0.485, 0.456, 0.406],
        'std': [0.229, 0.224, 0.225],
        'heads': ['vascular_disease', 'systemic_hypertension', 'dr_grade'],
        'dr_classes': DR_CLASSES,
    },
    'thresholds': {
        'vascular_disease': {
            'value': best_thresh,
            'source': f'Optimized via Youden J on validation set during {best_exp}',
            'label': 'Cardiovascular Disease (Vascular Disease)',
        },
        'systemic_hypertension': {
            'value': 0.4288,
            'source': 'Frozen validation threshold',
            'label': 'Systemic Hypertension',
        },
        'dr_confidence': {
            'min_top_probability': 0.50,
            'min_margin': 0.15,
        },
    },
    'evaluation_results': {
        'validation': best_rep['val_metrics'],
        'test': final_test_metrics,
    },
}

with open(export_dir / 'config_candidate.json', 'w') as f:
    json.dump(candidate_config, f, indent=2)

print(f'[SUCCESS] Candidate deployment package successfully exported to:')
print(f'  Directory: {export_dir}')
print(f'  Model    : {candidate_pt}')
print(f'  Config   : {export_dir / "config_candidate.json"}')
